In [ ]:
import pickle
import pyreadr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Define parameter names (matches our 10 camera parameters)
param_names = ["Canon", "Sony", "Nikon", "Panasonic", "Pixels", "Zoom", "Video", "Swivel", "Wifi", "Price"]
k_dim = len(param_names)

In [ ]:
# 1. Load Liesel Results
print("Loading Liesel samples...")
with open("liesel_posterior_samples_camera.pkl", "rb") as f:
    liesel_data = pickle.load(f)

liesel_samples = liesel_data["samples"]
liesel_mu = liesel_samples["mu"].reshape(-1, k_dim) # Shape: (Draws, 10)
liesel_beta = liesel_samples["beta_i"]              # Shape: (Draws, Units, 10)

# 2. Load Bayesm Results
print("Loading Bayesm samples...")
# Update this path if your R output saved somewhere else
rdata_path = "../HMBNL/camera/Data/bayesm_output_camera.RData" 
bayesm_rdata = pyreadr.read_r(rdata_path)

bayesm_mu = bayesm_rdata["mu_draws_df"].values      # Shape: (Draws, 10)
# Shape: (Units, 10, Draws) -> Transpose to (Draws, Units, 10) to match Liesel
bayesm_beta = np.transpose(bayesm_rdata["beta_reordered"].values, (2, 0, 1))

print(f"Liesel Mu Shape: {liesel_mu.shape}")
print(f"Bayesm Mu Shape: {bayesm_mu.shape}")

In [ ]:
# Calculate Summary Statistics
liesel_mu_mean = np.mean(liesel_mu, axis=0)
liesel_mu_std = np.std(liesel_mu, axis=0)

bayesm_mu_mean = np.mean(bayesm_mu, axis=0)
bayesm_mu_std = np.std(bayesm_mu, axis=0)

# Create a Comparison DataFrame
comp_df = pd.DataFrame({
    "Parameter": param_names,
    "Liesel Mean": liesel_mu_mean,
    "Bayesm Mean": bayesm_mu_mean,
    "Diff (Mean)": np.abs(liesel_mu_mean - bayesm_mu_mean),
    "Liesel Std": liesel_mu_std,
    "Bayesm Std": bayesm_mu_std,
    "Diff (Std)": np.abs(liesel_mu_std - bayesm_mu_std)
}).set_index("Parameter")

print("=== Comparison of Global Mean Utilities (\u03bc) ===")
display(comp_df.round(4))

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 20))
axes = axes.flatten()

for i in range(k_dim):
    ax = axes[i]
    
    # Plot Liesel
    sns.kdeplot(liesel_mu[:, i], ax=ax, fill=True, color="#1f77b4", label="Liesel (NUTS)", alpha=0.5)
    ax.axvline(np.mean(liesel_mu[:, i]), color="#1f77b4", linestyle="--")
    
    # Plot Bayesm
    sns.kdeplot(bayesm_mu[:, i], ax=ax, fill=True, color="#d62728", label="Bayesm (Gibbs/RW)", alpha=0.5)
    ax.axvline(np.mean(bayesm_mu[:, i]), color="#d62728", linestyle="--")
    
    ax.set_title(f"Posterior: {param_names[i]}", fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    
    if i == 0:
        ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate posterior means for unit-level betas
# Shape goes from (Draws, Units, 10) -> (Units, 10)
liesel_beta_means = np.mean(liesel_beta, axis=0)
bayesm_beta_means = np.mean(bayesm_beta, axis=0)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i in range(k_dim):
    ax = axes[i]
    
    x = bayesm_beta_means[:, i]
    y = liesel_beta_means[:, i]
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, color="#2ca02c", edgecolor='k')
    
    # 45-degree reference line
    min_val = min(np.min(x), np.min(y))
    max_val = max(np.max(x), np.max(y))
    ax.plot([min_val, max_val], [min_val, max_val], "k--", lw=2, label="45-degree line")
    
    # Correlation
    corr = np.corrcoef(x, y)[0, 1]
    
    ax.set_title(f"{param_names[i]}\nCorrelation: {corr:.3f}", fontweight="bold")
    ax.set_xlabel("Bayesm Posterior Mean")
    ax.set_ylabel("Liesel Posterior Mean")

axes[0].legend()
plt.tight_layout()
plt.show()